# 🌍 World Bank Data Analysis
### MCSE Banana-Level Problem

**Name:** HARSHITHA. B. T  
**SRN:** PES1UG24AM118  
**Section:** B (CSE – AIML)

---

## Objective
Fetch World Bank indicators (GDP, Population, Adult Literacy, CO₂ per capita), clean and preprocess the data, compute descriptive statistics and growth rates, visualize results, and draw meaningful inferences.

## Workflow
```
Web Scraping → Data Cleaning → Descriptive Statistics → Visualization → Inferences
```

## Datasets Used
| # | Indicator | Code | Description |
|---|-----------|------|-------------|
| 1 | GDP (current US$) | NY.GDP.MKTP.CD | Gross Domestic Product at current prices |
| 2 | Population, total | SP.POP.TOTL | Total population per country |
| 3 | Literacy rate (adult %) | SE.ADT.LITR.ZS | % of people aged 15+ who can read and write |
| 4 | CO₂ emissions (metric tons per capita) | EN.ATM.CO2E.PC | CO₂ emissions divided by midyear population |

## 1. Import Required Libraries

In [ ]:
# Install dependencies (run once in Colab)
!pip install beautifulsoup4 requests pandas matplotlib seaborn -q

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import numpy as np
import requests
import zipfile
import io
import warnings
warnings.filterwarnings('ignore')

# Plot styling
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.titlesize': 14,
    'axes.labelsize': 12
})

print('✅ Libraries imported successfully')

## 2. Indicator URLs — World Bank API

In [ ]:
# World Bank indicator codes
indicator_pages = {
    'GDP':        ('https://data.worldbank.org/indicator/NY.GDP.MKTP.CD',  'NY.GDP.MKTP.CD'),
    'Population': ('https://data.worldbank.org/indicator/SP.POP.TOTL',     'SP.POP.TOTL'),
    'Literacy':   ('https://data.worldbank.org/indicator/SE.ADT.LITR.ZS',  'SE.ADT.LITR.ZS'),
    'CO2':        ('https://data.worldbank.org/indicator/EN.ATM.CO2E.PC',  'EN.ATM.CO2E.PC'),
}

print('Indicators configured:')
for name, (page_url, code) in indicator_pages.items():
    print(f'  {name:12s}: {code}')

## 3. Web Scraping — Download from World Bank API

In [ ]:
datasets = {}

for name, (page_url, code) in indicator_pages.items():
    print(f'\nScraping {name} from {page_url} ...')

    # Build the World Bank CSV download URL
    csv_url = f'https://api.worldbank.org/v2/en/indicator/{code}?downloadformat=csv'
    print(f'Downloading ZIP from: {csv_url}')

    r = requests.get(csv_url, timeout=30)
    z = zipfile.ZipFile(io.BytesIO(r.content))

    # Find the main data CSV (starts with API_)
    data_file = [f for f in z.namelist() if f.startswith('API_') and f.endswith('.csv')][0]
    print(f'Extracting: {data_file}')

    with z.open(data_file) as f:
        df = pd.read_csv(f, skiprows=4)
        datasets[name] = df
        print(df.head(3))

print('\n✅ All datasets downloaded successfully')

## 4. Data Cleaning & Preprocessing

In [ ]:
cleaned_datasets = {}

for name, df in datasets.items():
    print(f'\nCleaning {name} dataset...')

    # Drop metadata columns not needed for analysis
    df = df.drop(columns=['Indicator Name', 'Indicator Code'], errors='ignore')

    # Melt wide format → long format (Country, Year, Value)
    df_long = df.melt(
        id_vars=['Country Name', 'Country Code'],
        var_name='Year',
        value_name=name
    )

    # Convert Year to numeric, drop non-year columns (e.g. 'Unnamed: 69')
    df_long['Year'] = pd.to_numeric(df_long['Year'], errors='coerce')
    df_long = df_long.dropna(subset=['Year'])
    df_long['Year'] = df_long['Year'].astype(int)

    # Drop rows where the indicator value is missing
    df_long = df_long.dropna(subset=[name])

    # ── FIX: CO2 data is already per capita from the World Bank API ──
    # EN.ATM.CO2E.PC is metric tons per capita — no transformation needed.
    # Previous versions accidentally used total emissions data.
    # Valid range: 0 – 50 metric tons per capita
    if name == 'CO2':
        before = len(df_long)
        df_long = df_long[(df_long[name] >= 0) & (df_long[name] <= 50)]
        after = len(df_long)
        print(f'  CO2: removed {before - after} out-of-range rows '
              f'(valid range: 0–50 metric tons per capita)')

    # Remove aggregate regions (keep only individual countries)
    # World Bank aggregate codes are 3-char and match known patterns
    aggregate_codes = [
        'WLD','HIC','LMC','UMC','LIC','EAP','ECA','LAC','MNA','NAC',
        'SAS','SSA','AFE','AFW','ARB','CEB','CSS','EAR','EAS','EMU',
        'EUU','FCS','HPC','IBD','IBT','IDA','IDB','IDX','LAC','LCN',
        'LDC','LMY','MEA','MIC','OED','OSS','PRE','PSS','PST','SAS',
        'SSF','SST','TEA','TEC','TLA','TMN','TSA','TSS'
    ]
    df_long = df_long[~df_long['Country Code'].isin(aggregate_codes)]

    cleaned_datasets[name] = df_long
    print(f'  Shape: {df_long.shape}  |  '
          f'Years: {df_long["Year"].min()}–{df_long["Year"].max()}  |  '
          f'Countries: {df_long["Country Name"].nunique()}')
    print(df_long.head(3))

print('\n✅ All datasets cleaned')

## 5. Descriptive Statistics

In [ ]:
desc_stats = {}

print('=' * 60)
for name, df in cleaned_datasets.items():
    stats = df[name].describe()
    desc_stats[name] = stats
    print(f'\nDescriptive Statistics — {name}:')
    print(stats.to_string())
    print('-' * 40)
print('=' * 60)

In [ ]:
# Summary table — all four indicators side by side
summary_rows = ['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']
summary_df = pd.DataFrame({
    name: desc_stats[name].loc[summary_rows]
    for name in desc_stats
})

# Format for readability
pd.options.display.float_format = '{:,.2f}'.format
print('\nCombined Descriptive Statistics Summary:')
print(summary_df.to_string())

## 6. Growth Rate Analysis

In [ ]:
# Compute compound annual growth rate (CAGR) for India, China, US
sample_countries = ['India', 'China', 'United States']
year_start, year_end = 1990, 2020

print(f'CAGR ({year_start}–{year_end}) for India, China, United States')
print('=' * 60)

for name, df in cleaned_datasets.items():
    print(f'\n{name}:')
    for country in sample_countries:
        subset = df[
            (df['Country Name'] == country) &
            (df['Year'].isin([year_start, year_end]))
        ].sort_values('Year')

        if len(subset) == 2:
            v_start = subset.iloc[0][name]
            v_end   = subset.iloc[1][name]
            n_years = year_end - year_start
            if v_start > 0 and v_end > 0:
                cagr = ((v_end / v_start) ** (1 / n_years) - 1) * 100
                print(f'  {country:15s}: {cagr:+.2f}% per year')
            else:
                print(f'  {country:15s}: N/A (zero or negative base value)')
        else:
            print(f'  {country:15s}: insufficient data')

## 7. Visualizations

For each indicator: Histogram · Boxplot · Line plot (India, China, US trends over time)

In [ ]:
sample_countries = ['India', 'China', 'United States']

# ── GDP ──────────────────────────────────────────────────────
df_gdp = cleaned_datasets['GDP']

# Histogram
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(df_gdp['GDP'], bins=60, kde=True, ax=ax, color='steelblue')
ax.set_title('Distribution of GDP (current US$) — All Countries, All Years')
ax.set_xlabel('GDP (US$)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e12:.0f}T' if x >= 1e12 else f'${x/1e9:.0f}B'))
plt.tight_layout()
plt.savefig('gdp_hist.png', dpi=120)
plt.show()

# Boxplot
fig, ax = plt.subplots(figsize=(10, 4))
sns.boxplot(x=df_gdp['GDP'], ax=ax, color='steelblue')
ax.set_title('GDP Spread and Outliers')
ax.set_xlabel('GDP (US$)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e12:.0f}T' if x >= 1e12 else f'${x/1e9:.0f}B'))
plt.tight_layout()
plt.savefig('gdp_box.png', dpi=120)
plt.show()

# Line plot — India, China, US
df_sample = df_gdp[df_gdp['Country Name'].isin(sample_countries)]
fig, ax = plt.subplots(figsize=(12, 5))
sns.lineplot(data=df_sample, x='Year', y='GDP', hue='Country Name', linewidth=2, ax=ax)
ax.set_title('GDP Trends Over Time — India, China, United States')
ax.set_ylabel('GDP (US$)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e12:.1f}T'))
ax.legend(title='Country')
plt.tight_layout()
plt.savefig('gdp_line.png', dpi=120)
plt.show()

In [ ]:
# ── POPULATION ───────────────────────────────────────────────
df_pop = cleaned_datasets['Population']

# Histogram
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(df_pop['Population'], bins=60, kde=True, ax=ax, color='teal')
ax.set_title('Distribution of Population — All Countries, All Years')
ax.set_xlabel('Population')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e9:.1f}B' if x >= 1e9 else f'{x/1e6:.0f}M'))
plt.tight_layout()
plt.savefig('population_hist.png', dpi=120)
plt.show()

# Boxplot
fig, ax = plt.subplots(figsize=(10, 4))
sns.boxplot(x=df_pop['Population'], ax=ax, color='teal')
ax.set_title('Population Spread and Outliers')
ax.set_xlabel('Population')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e9:.1f}B' if x >= 1e9 else f'{x/1e6:.0f}M'))
plt.tight_layout()
plt.savefig('population_box.png', dpi=120)
plt.show()

# Line plot — India, China, US
df_sample = df_pop[df_pop['Country Name'].isin(sample_countries)]
fig, ax = plt.subplots(figsize=(12, 5))
sns.lineplot(data=df_sample, x='Year', y='Population', hue='Country Name', linewidth=2, ax=ax)
ax.set_title('Population Trends Over Time — India, China, United States')
ax.set_ylabel('Population')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e9:.1f}B'))
ax.legend(title='Country')
plt.tight_layout()
plt.savefig('population_line.png', dpi=120)
plt.show()

In [ ]:
# ── LITERACY ─────────────────────────────────────────────────
df_lit = cleaned_datasets['Literacy']

# Histogram
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(df_lit['Literacy'], bins=50, kde=True, ax=ax, color='darkorange')
ax.set_title('Distribution of Adult Literacy Rate — All Countries, All Years')
ax.set_xlabel('Literacy Rate (%)')
plt.tight_layout()
plt.savefig('literacy_hist.png', dpi=120)
plt.show()

# Boxplot
fig, ax = plt.subplots(figsize=(10, 4))
sns.boxplot(x=df_lit['Literacy'], ax=ax, color='darkorange')
ax.set_title('Literacy Rate Spread and Outliers')
ax.set_xlabel('Literacy Rate (%)')
plt.tight_layout()
plt.savefig('literacy_box.png', dpi=120)
plt.show()

# Line plot — India, China, US
# FIX: explicitly filter all three countries and handle missing US data
df_sample = df_lit[df_lit['Country Name'].isin(sample_countries)].copy()
available = df_sample['Country Name'].unique()
print(f'Countries with literacy data: {list(available)}')

fig, ax = plt.subplots(figsize=(12, 5))
sns.lineplot(data=df_sample, x='Year', y='Literacy', hue='Country Name', linewidth=2, ax=ax)
ax.set_title('Adult Literacy Rate Trends — India, China, United States')
ax.set_ylabel('Literacy Rate (%)')
ax.set_ylim(0, 105)
ax.legend(title='Country')

# Annotate if US data is absent (World Bank doesn't always track US literacy)
if 'United States' not in available:
    ax.text(0.5, 0.05,
            'Note: US literacy data not available in World Bank dataset\n'
            '(US reports ~99% through other sources)',
            transform=ax.transAxes, ha='center', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig('literacy_line.png', dpi=120)
plt.show()

In [ ]:
# ── CO₂ EMISSIONS (PER CAPITA) ───────────────────────────────
# FIX: Using EN.ATM.CO2E.PC — metric tons per capita (correct indicator)
# Values should be in range 0–50. Data cleaned in Section 4.
df_co2 = cleaned_datasets['CO2']

print(f'CO2 data range: {df_co2["CO2"].min():.2f} – {df_co2["CO2"].max():.2f} metric tons per capita')

# Histogram
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(df_co2['CO2'], bins=50, kde=True, ax=ax, color='crimson')
ax.set_title('Distribution of CO₂ Emissions (metric tons per capita) — All Countries, All Years')
ax.set_xlabel('CO₂ Emissions (metric tons per capita)')
plt.tight_layout()
plt.savefig('CO2_hist.png', dpi=120)
plt.show()

# Boxplot
fig, ax = plt.subplots(figsize=(10, 4))
sns.boxplot(x=df_co2['CO2'], ax=ax, color='crimson')
ax.set_title('CO₂ Emissions Spread and Outliers (metric tons per capita)')
ax.set_xlabel('CO₂ Emissions (metric tons per capita)')
plt.tight_layout()
plt.savefig('CO2_box.png', dpi=120)
plt.show()

# Line plot — India, China, US
df_sample = df_co2[df_co2['Country Name'].isin(sample_countries)]
fig, ax = plt.subplots(figsize=(12, 5))
sns.lineplot(data=df_sample, x='Year', y='CO2', hue='Country Name', linewidth=2, ax=ax)
ax.set_title('CO₂ Emissions Trends (metric tons per capita) — India, China, United States')
ax.set_ylabel('CO₂ (metric tons per capita)')
ax.legend(title='Country')
plt.tight_layout()
plt.savefig('CO2_line.png', dpi=120)
plt.show()

## 8. Inferences

In [ ]:
inferences = """
========================================================
INFERENCES FROM WORLD BANK DATA ANALYSIS
========================================================

1. GDP (current US$):
   - Distribution is highly right-skewed: most countries have small GDPs
     while USA and China dominate at the extreme high end.
   - Boxplot confirms massive outliers — USA and China are visible far
     beyond the IQR range.
   - Line plot: China rises sharply post-1990s due to economic reforms;
     India grows steadily; USA remains consistently highest overall.
   - Takeaway: Global GDP is extremely unequal across nations.

2. Population, total:
   - Most countries have small populations; India and China are extreme
     outliers at 1.4B+ each.
   - Boxplot shows the median is well below 50 million, confirming
     right-skewed distribution.
   - Line plot: India surpassed China around 2023; US grows slowly
     and linearly in comparison.
   - Takeaway: Global population is concentrated in very few countries.

3. Literacy rate (adult %):
   - Distribution clusters toward higher values (80–100%) — global
     literacy has improved significantly since 1960.
   - Some outliers remain below 40%, representing countries with
     persistent educational challenges.
   - Line plot: India shows strong improvement from ~40% in 1980 to
     ~75%+ by 2020; China improved from ~65% to ~97%.
   - Note: US literacy is not tracked by World Bank (assumed ~99%
     through domestic sources).
   - Takeaway: Developing nations have made remarkable literacy gains
     but gaps persist.

4. CO₂ emissions (metric tons per capita):
   - Distribution is right-skewed: most countries emit under 5 tons
     per capita; developed and oil-rich nations are outliers.
   - Boxplot confirms skew with outliers like Qatar (>30 tons/capita).
   - Line plot: USA consistently high (~15–20 tons); China rose sharply
     post-2000 (~8 tons); India remains low (~2 tons) reflecting its
     developing economy status.
   - Takeaway: Per-capita CO₂ responsibility falls disproportionately
     on developed nations, despite China's recent rise.

========================================================
Key Learning from this Banana Problem:
   - Real-world data is always messy — cleaning is 60% of the work.
   - Using the WRONG indicator (total vs per capita) completely changes
     the analysis and inferences drawn.
   - Aggregate regions (World Bank regional codes) must be filtered
     out before country-level analysis.
   - Visualization makes patterns invisible in raw numbers immediately
     obvious (e.g., China's GDP inflection point post-1990).
========================================================
"""
print(inferences)

---
## Summary

| Indicator | Key Finding |
|-----------|-------------|
| GDP | Highly skewed; China surging, USA dominant |
| Population | India & China extreme outliers; India now #1 |
| Literacy | Global improvement; India from 40% to 75%+ |
| CO₂/capita | USA historically highest; India lowest among 3 |

**Thank You!**  
*HARSHITHA. B. T | PES1UG24AM118 | CSE-AIML B*